# Submit Training Job to Azure ML

This notebook submits the fine-tuning job to Azure ML GPU clusters for efficient training.

## What This Notebook Does

1. **Configures Training Environment**: Creates conda environment with PyTorch, transformers, PEFT
2. **Prepares Training Script**: Packages training code and dependencies for remote execution
3. **Submits GPU Job**: Launches training on auto-scaling GPU compute cluster
4. **Monitors Progress**: Tracks training metrics, loss curves, and resource utilization
5. **Manages Experiments**: Organizes training runs with versioning and metadata

## Why Use Azure ML for Training?

**GPU Acceleration:**
- Fine-tuning on CPU: ~20-40 hours
- Fine-tuning on single V100 GPU: ~2-4 hours
- 10-20x speedup makes experimentation practical

**Auto-scaling:**
- Cluster scales to 0 when idle (no cost)
- Scales up when job submitted (automatic provisioning)
- Scales down after job completes (saves money)

**Experiment Tracking:**
- Automatic logging of metrics, hyperparameters, artifacts
- Compare multiple training runs side-by-side
- Reproducible with versioned data, code, and environment

**Managed Infrastructure:**
- No GPU driver installation or CUDA configuration
- Pre-built environments with PyTorch/TensorFlow
- Automatic retry on transient failures

## Training Job Components

### Training Script
Located at `src/training/train.py`, handles:
- Data loading and preprocessing
- Model initialization with LoRA adapters
- Training loop with gradient accumulation
- Checkpoint saving every N steps
- Metric logging to Azure ML

### Compute Cluster
- VM Type: Standard_NC6s_v3 (1x V100 GPU, 16GB GPU RAM)
- Auto-scale: 0-4 nodes
- Idle timeout: 120 seconds
- Cost: ~$3.06/hour per node (only when running)

### Environment
- Base image: `mcr.microsoft.com/azureml/curated/acft-hf-nlp-gpu`
- Python 3.11
- PyTorch 2.1+ with CUDA 11.8
- Transformers, PEFT, datasets, accelerate

## Training Monitoring

### Real-time Metrics
- Training loss (per step)
- Validation loss (per epoch)
- Learning rate schedule
- GPU utilization
- Memory usage

### Outputs
- Model checkpoints (saved every 500 steps)
- Final trained model
- Training logs
- Metric history (JSON)

## Cost Estimation

| Configuration | Duration | Cost |
|---------------|----------|------|
| Small dataset (<10K examples) | 1-2 hours | $3-6 |
| Medium dataset (10-50K examples) | 2-4 hours | $6-12 |
| Large dataset (50K+ examples) | 4-8 hours | $12-24 |

## Prerequisites

- Completed notebook `03-provision-compute.ipynb`
- Completed notebook `05-train-model.ipynb` (validated training pipeline locally)
- GPU compute cluster provisioned and ready
- Training data uploaded to Azure Blob Storage
- Sufficient GPU quota in subscription

## Expected Duration

- Job submission: ~2-5 minutes
- Training: 1-8 hours depending on dataset size
- This notebook completes immediately after submission; training runs asynchronously

## 1. Setup Azure ML Workspace

**Why use JobManager?** The `AzureMLJobManager` class abstracts Azure ML SDK complexity, handling environment creation, job configuration, script packaging, and error handling with sensible defaults.

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from src.training.job_manager import AzureMLJobManager
import yaml

# Initialize Azure ML client
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

print(f"✓ Connected to workspace: {ml_client.workspace_name}")
print(f"  Subscription: {ml_client.subscription_id}")
print(f"  Resource Group: {ml_client.resource_group_name}")

# Initialize job manager
job_manager = AzureMLJobManager()
print("✓ Job manager initialized")

## 2. Verify Training Configuration

In [ ]:
# Load training configuration
config_path = project_root / "configs" / "training_config.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

print("Training Configuration:")
print(f"  Model: {config['model']['name_or_path']}")
print(f"  Epochs: {config['training']['num_epochs']}")
print(f"  Batch size: {config['training']['per_device_train_batch_size']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  LoRA rank: {config['lora']['r']}")
print(f"  LoRA alpha: {config['lora']['alpha']}")
print(f"\nCompute:")
compute_name = config.get('azure_ml', {}).get('compute_target', 'gpu-cluster')
print(f"  Cluster: {compute_name}")

## 3. Create Training Environment

**Why create environment?** Azure ML environments define the Docker container with all dependencies (PyTorch, CUDA, transformers) that runs on the compute cluster. We version environments to ensure reproducibility.

In [ ]:
# Create or update Azure ML environment
environment_name = "phi4-training-env"
conda_file = str(project_root / "configs" / "conda.yaml")

print(f"Creating environment: {environment_name}")
environment = job_manager.create_environment(
    name=environment_name,
    conda_file=conda_file,
    description="PyTorch environment for Phi-4 fine-tuning with LoRA",
)

print(f"✓ Environment created: {environment.name}")
print(f"  Version: {environment.version}")

## 4. Verify Compute Cluster

In [ ]:
# Check compute cluster status
try:
    compute = ml_client.compute.get(compute_name)
    print(f"✓ Compute cluster found: {compute.name}")
    print(f"  Type: {compute.type}")
    print(f"  Size: {compute.size}")
    print(f"  State: {compute.provisioning_state}")
    print(f"  Current nodes: {compute.current_node_count if hasattr(compute, 'current_node_count') else 'N/A'}")
except Exception as e:
    print(f"⚠️  Compute cluster not found: {e}")
    print("Please run notebook 03 to provision compute cluster.")

## 5. Submit Training Job

In [ ]:
# Configure job parameters
experiment_name = config.get('azure_ml', {}).get('experiment_name', 'phi-4-training')
display_name = f"phi4-finetuning-{pd.Timestamp.now().strftime('%Y%m%d-%H%M%S')}"

# Command to run
command_str = "python src/training/train.py --config configs/training_config.yaml"

print(f"Submitting training job...")
print(f"  Experiment: {experiment_name}")
print(f"  Display name: {display_name}")
print(f"  Compute: {compute_name}")
print(f"  Environment: {environment_name}")
print(f"\nCommand: {command_str}")
print("\n" + "="*60)

# Submit job
job = job_manager.submit_training_job(
    experiment_name=experiment_name,
    display_name=display_name,
    code_path=str(project_root),
    command_str=command_str,
    environment_name=environment_name,
    compute_name=compute_name,
    environment_variables={
        "PYTORCH_CUDA_ALLOC_CONF": "max_split_size_mb:512",
    },
)

print(f"\n✓ Job submitted successfully!")
print(f"  Job ID: {job.name}")
print(f"  Status: {job.status}")
print(f"\n📊 View in Azure ML Studio:")
print(f"  {job.studio_url}")

## 6. Monitor Training Progress

In [ ]:
# Get current job status
job_name = job.name

status = job_manager.get_job_status(job_name)
print(f"Job Status:")
print(f"  Name: {status['name']}")
print(f"  Status: {status['status']}")
print(f"  Created: {status['creation_time']}")
print(f"  Duration: {status['duration']}")
print(f"\n🔗 Studio URL: {status['studio_url']}")

In [ ]:
# Wait for job completion (optional - can take 1-2 hours)
# Uncomment to wait for completion

# print("Waiting for job to complete...")
# print("This may take 1-2 hours depending on data size and configuration.")
# print("You can safely interrupt this cell and check status later.\n")

# final_status = job_manager.wait_for_completion(
#     job_name=job_name,
#     timeout_seconds=7200,  # 2 hours
#     check_interval=60,  # Check every minute
# )

# print(f"\n✓ Job completed with status: {final_status}")

## 7. Check Recent Training Jobs

In [ ]:
import pandas as pd

# List recent jobs
jobs = job_manager.list_jobs(experiment_name=experiment_name, max_results=5)

print(f"Recent jobs in '{experiment_name}':")
print("\n")

jobs_data = []
for j in jobs:
    jobs_data.append({
        "Name": j.name,
        "Display Name": j.display_name,
        "Status": j.status,
        "Created": j.creation_context.created_at.strftime("%Y-%m-%d %H:%M"),
    })

if jobs_data:
    df = pd.DataFrame(jobs_data)
    print(df.to_string(index=False))
else:
    print("No jobs found in this experiment.")

## 8. View MLflow Metrics (After Job Completes)

In [ ]:
# Get job metrics
# Uncomment after job completes

# metrics = job_manager.get_job_metrics(job_name)

# if metrics:
#     print("Training Metrics:")
#     for key, value in metrics.items():
#         print(f"  {key}: {value}")
# else:
#     print("No metrics available yet. Job may still be running.")

## 9. Download Job Outputs (After Completion)

In [ ]:
# Download outputs from completed job
# Uncomment after job completes successfully

# output_path = project_root / "outputs" / "downloaded" / job_name

# print(f"Downloading job outputs to: {output_path}")
# downloaded_path = job_manager.download_job_outputs(
#     job_name=job_name,
#     output_path=str(output_path)
# )

# print(f"\n✓ Outputs downloaded to: {downloaded_path}")

# # List downloaded files
# print("\nDownloaded files:")
# for file_path in downloaded_path.rglob("*"):
#     if file_path.is_file():
#         size_mb = file_path.stat().st_size / (1024 * 1024)
#         print(f"  {file_path.relative_to(downloaded_path)} ({size_mb:.1f} MB)")

## 10. Register Trained Model

In [ ]:
# Register model in Azure ML model registry
# Uncomment after job completes successfully

# model_name = "phi-4-finetuned"

# print(f"Registering model: {model_name}")
# registered_model = job_manager.register_model_from_job(
#     job_name=job_name,
#     model_name=model_name,
#     model_path="outputs/best_model",
#     description="Phi-4 fine-tuned with LoRA",
#     tags={
#         "framework": "pytorch",
#         "task": "text-generation",
#         "base_model": "phi-4",
#         "training_method": "lora",
#     },
# )

# print(f"\n✓ Model registered successfully!")
# print(f"  Name: {registered_model.name}")
# print(f"  Version: {registered_model.version}")
# print(f"  ID: {registered_model.id}")

## Next Steps

After training completes:
1. **Evaluate Model** (notebook 07): Run evaluation metrics on test set
2. **Optimize Model** (notebook 08): Apply quantization and ONNX conversion
3. **Build Container** (notebook 09): Package model in optimized container

## Troubleshooting

### Job Failed - Out of Memory (OOM)
Edit `configs/training_config.yaml`:
```yaml
training:
  per_device_train_batch_size: 1  # Reduce from 4
  gradient_accumulation_steps: 16  # Increase from 4
  use_4bit: true  # Enable 4-bit quantization
```

### Job Failed - Quota Exceeded
- Check Azure subscription GPU quota
- Request quota increase in Azure Portal
- Or use smaller VM size in compute cluster

### Job Stuck in Queue
- Check compute cluster has available capacity
- Verify cluster min_nodes > 0 for immediate availability
- Check for quota or networking issues

### Environment Build Failed
- Verify conda.yaml dependencies are compatible
- Check Azure ML environment build logs in Studio
- Try using a pre-built AzureML environment

### Training is Very Slow
- Verify GPU is being used (not CPU)
- Check `dataloader_num_workers` (try 4-8)
- Enable mixed precision: `bf16: true`
- Reduce logging frequency: `logging_steps: 50`